# HireBase API Bug / Anomalies Report & Developer Experience Feedback

During the evaluation and implementation of the data pipeline for the revolving door analysis, I ran into a few minor friction points with the developer experience and documentation:

## 1. Missing Authentication Header in Code Snippets
- **Issue:** Received a `401 Unauthorized` / `"Missing authorization header"` error when initially running the script.
- **Description:** The provided API documentation and/or boilerplate code snippets for the `v2/jobs/export` endpoint defined the API key but did not actually inject it into the request headers. We had to debug the request and manually configure the `headers={"X-API-Key": api_key, "Content-Type": "application/json"}` dictionary and pass it to the `requests.post()` and `requests.get()` functions.
- **Suggestion:** Please update the Python code snippets in the documentation to construct and pass the headers correctly so that the examples work seamlessly out-of-the-box.

In [1]:
import requests

url = "https://api.hirebase.org/v2/jobs/export"

payload = {
    "search": {
        "job_titles": ["<string>"],
        "keywords": ["<string>"],
        "job_board": "<string>",
        "job_slug": "<string>",
        "location_group": "<string>",
        "location_types": ["<string>"],
        "locations": [
            {
                "city": "<string>",
                "region": "<string>",
                "country": "<string>"
            }
        ],
        "experience": ["<string>"],
        "yoe": {
            "min": 123,
            "max": 123
        },
        "company_name": "<string>",
        "company_slug": "<string>",
        "salary": {
            "min": 123,
            "max": 123
        },
        "job_type": ["<string>"],
        "industry": ["<string>"],
        "sub_industry": ["<string>"],
        "date_posted": "<string>",
        "visa": "<string>",
        "sort_by": "<string>",
        "sort_order": "<string>",
        "currency": "<string>",
        "days_ago": 123,
        "include_expired": "<string>",
        "include_no_salary": "<string>",
        "include_yoe": "<string>",
        "month": "<string>",
        "page": 123,
        "limit": 123
    },
    "format": "<string>"
}
headers = {"Content-Type": "application/json"}

response = requests.post(url, json=payload, headers=headers)

print(response.text)

{"detail":"401: Missing authorization header"}


## 2. Prerequisite Missing from Documentation
- **Issue:** Encountered `ModuleNotFoundError: No module named 'requests'`
- **Description:** The Python examples rely on the external `requests` library to make HTTP calls. This isn't part of the standard Python library.
- **Suggestion:** It would be highly beneficial to add a small prerequisites note (e.g., `pip install requests`) right above the Python examples to assist newer developers or those setting up a fresh environment.

## 3. Asynchronous Export Task Polling
- **Issue:** Managing task state (`finished` vs `completed`)
- **Description:** When polling the `/v2/tasks/{task_id}` endpoint, I had to implement logic to handle multiple possible completion states and delays. 
- **Suggestion:** Providing a lightweight official Python SDK or wrapping the polling mechanism in a helper function within the documentation would improve the developer experience immensely and prevent every developer from re-inventing the same polling loops.

As part of the exploratory analysis, I encountered anomalies in the job postings, missing crucial fields like `date_posted`, `company_name`, `job_title`.

In [11]:
from pandas import read_csv
import pandas as pd
import os
import json

jobs = []
with open("exported_jobs.json", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            jobs.append(json.loads(line))

df = pd.DataFrame(jobs)
anomalies_df = pd.read_csv("anomaly_ids.csv")
df[df['_id'].isin(anomalies_df['_id'])]


,_id,company_name,job_title,application_link,location_raw,job_type,date_posted,company_logo,job_board,job_board_link,...,meta,score,team,salary_range,technologies,skills,benefits,contact_email,contact_phone,expired
16,690fe0be406265b3de6e60db,Sfermion,NaN,https://sfermion.breezy.hr/p/234a41fa5d69-exec...,NaN,Full Time,2025-11-09,https://media.licdn.com/dms/image/v2/D560BAQHh...,breezyhr,https://sfermion.breezy.hr,...,CDMaAmABIg5aDGkP4L5AYmWz3m5g2w==,4.0,executive team,NaN,"[Gmail, Google Calendar, Word, Excel, PowerPoi...","[calendar management, prioritization, communic...",NaN,NaN,NaN,NaN
285,690ffecc406265b3de6e702e,Plat4mation,NaN,https://plat4mation.breezy.hr/p/7631c69f66de01...,NaN,Full Time,2025-11-09,https://media.licdn.com/dms/image/v2/C560BAQHr...,breezyhr,https://plat4mation.breezy.hr,...,CKgGGgJgASIOWgxpD/7MQGJls95ucC4=,4.0,NaN,NaN,"[JavaScript, Python, Shell, PERL, Web service ...","[Platform-as-a-Service (PaaS) technology, Serv...",NaN,NaN,NaN,NaN
286,690ffecc406265b3de6e7031,Plat4mation,NaN,https://plat4mation.breezy.hr/p/4057ced44c4e01...,NaN,Full Time,2025-11-09,https://media.licdn.com/dms/image/v2/C560BAQHr...,breezyhr,https://plat4mation.breezy.hr,...,CKsGGgJgASIOWgxpD/7MQGJls95ucDE=,4.0,Managed Services team,NaN,[ServiceNow],"[Leadership, Communication, Problem-solving, T...","[Generous Paid Time Off, 401k Matching, Retire...",NaN,NaN,NaN
287,690ffecc406265b3de6e7032,Plat4mation,NaN,https://plat4mation.breezy.hr/p/d1caca8a40ed01...,NaN,Full Time,2025-11-09,https://media.licdn.com/dms/image/v2/C560BAQHr...,breezyhr,https://plat4mation.breezy.hr,...,CKwGGgJgASIOWgxpD/7MQGJls95ucDI=,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
288,690ffecc406265b3de6e7035,Plat4mation,NaN,https://plat4mation.breezy.hr/p/931a74bc186901...,NaN,Full Time,2025-11-09,https://media.licdn.com/dms/image/v2/C560BAQHr...,breezyhr,https://plat4mation.breezy.hr,...,CK4GGgJgASIOWgxpD/7MQGJls95ucDU=,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99073,692ed94a08ee7dec96c19c04,NaN,Client Partner,https://talent.faktion.com/o/client-partner,"Antwerp, Vlaams Gewest, Belgium",Full Time,2025-06-27,NaN,recruitee,https://faktionbv1.recruitee.com,...,CNCkCRoCYAEiDloMaS7ZSgjufeyWwZwE,4.0,NaN,None,"[AI, CRM tools]","[Account management, Consulting, Technology ad...",[Generous Paid Time Off],NaN,NaN,NaN
99114,692edbb408ee7dec96c19ff0,NaN,EMPLOYE(E) DE CORDONNERIE,https://groupepontdebois.recruitee.com/o/emplo...,"Saint-Renan, Bretagne, France",Full Time,2025-10-10,NaN,recruitee,https://groupepontdebois.recruitee.com,...,CKalCRoCYAEiDloMaS7btAjufeyWwZ/w,4.0,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN
99572,692f49f15ceec26d80d6af3e,NaN,Sales & Marketing Coordinator (m/w/d),https://rscinternationalgmbh.recruitee.com/o/s...,"Dresden, Sachsen, Germany",Part Time,2025-08-29,NaN,recruitee,https://rscinternationalgmbh.recruitee.com,...,CIqtCRoCYAEiDloMaS9J8Vzuwm2A1q8+,4.0,NaN,None,"[Office Tools, CRM systems]","[Sales 2.0, CRM system, German language]",NaN,NaN,NaN,NaN
99574,692f49f15ceec26d80d6afd0,NaN,Stage Communicatie,https://werkenbij.kader.nl/o/stage-communicatie,"Zeist, Utrecht, Nederland",Internship,2025-09-26,NaN,recruitee,https://kadergroup.recruitee.com,...,CI+tCRoCYAEiDloMaS9J8Vzuwm2A1q/Q,4.0,Marketing and Communications team,None,NaN,NaN,NaN,NaN,NaN,NaN
